# Tokens: AI text kaise padhta hai?
**AI Explained · Episode 1 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Tokenizer: `tiktoken` (OpenAI ka open-source tokenizer). `o200k_base` = GPT-4o family, `cl100k_base` = GPT-4.

In [1]:
!pip install -q tiktoken

In [2]:
import tiktoken

# GPT-4o wala tokenizer
enc = tiktoken.get_encoding("o200k_base")

In [3]:
text = "AI makes life easier!"
ids = enc.encode(text)
print(ids)
print(len(ids), "tokens")

[17527, 4787, 2615, 11087, 0]
5 tokens


In [4]:
print(enc.decode(ids))

AI makes life easier!


In [5]:
def show(text, e=enc):
    ids = e.encode(text)
    parts = [e.decode([i]) for i in ids]
    print(len(ids), "tokens:", parts)

show("AI makes life easier!")

5 tokens: ['AI', ' makes', ' life', ' easier', '!']


In [6]:
show("AI जीवन को आसान बनाता है!")

8 tokens: ['AI', ' जीवन', ' को', ' आसान', ' बन', 'ाता', ' है', '!']


In [7]:
# GPT-4 wala purana tokenizer
old = tiktoken.get_encoding("cl100k_base")
show("AI जीवन को आसान बनाता है!", old)

24 tokens: ['AI', ' �', '�', 'ी', '�', '�', 'न', ' क', 'ो', ' �', '�', 'स', 'ा�', '�', ' �', '�', 'न', 'ा�', '�', 'ा', ' ह', '�', '�', '!']


In [8]:
print(list("ज".encode("utf-8")))
print(old.n_vocab, enc.n_vocab)

[224, 164, 156]
100277 200019


In [9]:
hi = "AI जीवन को आसान बनाता है!"
ro = "AI jeevan ko aasaan banata hai!"
for e in (old, enc):
    print(e.name, len(e.encode(hi)), len(e.encode(ro)))

cl100k_base 24 12
o200k_base 8 11


In [10]:
show("2026")
show("123456789")
show("hello Hello HELLO")

2 tokens: ['202', '6']
3 tokens: ['123', '456', '789']
4 tokens: ['hello', ' Hello', ' HEL', 'LO']


In [11]:
show("strawberry")
show("I love strawberry")
print("strawberry".count("r"))

3 tokens: ['st', 'raw', 'berry']
3 tokens: ['I', ' love', ' strawberry']
3


## Tokenizer khud kaise seekhta hai? (BPE)
Byte Pair Encoding: sabse zyada saath aane wala jodaa (pair) dhoondho, use ek token bana do, phir repeat.

In [12]:
from collections import Counter

corpus = "padhna padhai padhega likhna likhai likhega"
seqs = [list(w) for w in corpus.split()]

def top_pair(seqs):
    c = Counter()
    for s in seqs:
        c.update(zip(s, s[1:]))
    return c.most_common(1)[0]

print(top_pair(seqs))

(('p', 'a'), 3)


In [13]:
def merge(s, a, b):
    out, i = [], 0
    while i < len(s):
        if s[i:i + 2] == [a, b]:
            out.append(a + b)
            i += 2
        else:
            out.append(s[i])
            i += 1
    return out

for step in range(1, 11):
    (a, b), n = top_pair(seqs)
    seqs = [merge(s, a, b) for s in seqs]
    print(step, a + b, n)
print(seqs)

1 pa 3
2 pad 3
3 padh 3
4 li 3
5 lik 3
6 likh 3
7 na 2
8 ai 2
9 eg 2
10 ega 2
[['padh', 'na'], ['padh', 'ai'], ['padh', 'ega'], ['likh', 'na'], ['likh', 'ai'], ['likh', 'ega']]


In [14]:
en = "Tomorrow is my exam, please make a study plan."
hi = "कल मेरी परीक्षा है, कृपया पढ़ाई की योजना बनाओ।"
for e in (old, enc):
    a, b = len(e.encode(en)), len(e.encode(hi))
    print(e.name, a, b, round(b / a, 1), "x")

cl100k_base 11 48 4.4 x
o200k_base 11 15 1.4 x


In [15]:
# Apna sentence daalo aur chalao:
show("मुझे AI सीखना है")

6 tokens: ['म', 'ुझे', ' AI', ' सीख', 'ना', ' है']
